# HealthConnect Clinic — Week 5 ML Pipeline Demonstration
**Track:** Machine Learning Engineering
**Intern:** Marie Claire Niyomugenga
**Project:** HealthConnect Appointment No-Show Prediction — Experience Lab

This notebook demonstrates the Week 5 pipeline end-to-end: data ingestion & validation →
feature engineering → target definition → patient-grouped train/test split → baseline
model training → evaluation → inference.

**Key Week 5 outcome:** a Week 4 assumption was corrected here — `waiting_time_minutes`
was originally excluded as a suspected data-leakage field, but the Data Dictionary labels
it "Estimated waiting time," and it's populated across all three appointment outcomes
(Attended, Cancelled, No-Show) with nearly identical means — confirming it's a
pre-appointment estimate, not a leakage risk. It has been included in the feature set
accordingly (see the check early in this notebook).

Baseline model: `LogisticRegression`, ROC-AUC 0.683 on a patient-grouped held-out test set.

In [1]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Setup complete.")

Setup complete.


In [2]:
from google.colab import files

print("Upload HealthConnect_Appointment_Data.csv and HealthConnect_Data_Dictionary.xlsx")
uploaded = files.upload()

Upload HealthConnect_Appointment_Data.csv and HealthConnect_Data_Dictionary.xlsx


Saving HealthConnect_Appointment_Data (1).csv to HealthConnect_Appointment_Data (1).csv
Saving HealthConnect_Data_Dictionary.xlsx to HealthConnect_Data_Dictionary (1).xlsx


In [3]:
import os
for f in os.listdir():
    if f.endswith('.csv') or f.endswith('.xlsx'):
        print(f, "-", os.path.getsize(f), "bytes")

HealthConnect_Data_Dictionary (1).xlsx - 6448 bytes
HealthConnect_Data_Dictionary.xlsx - 6448 bytes
HealthConnect_Appointment_Data.csv - 590861 bytes
HealthConnect_Appointment_Data (1).csv - 590861 bytes


In [4]:
import os

if os.path.exists("HealthConnect_Appointment_Data (1).csv"):
    os.rename("HealthConnect_Appointment_Data (1).csv", "HealthConnect_Appointment_Data.csv")

for f in os.listdir():
    if f.endswith('.csv') or f.endswith('.xlsx'):
        print(f, "-", os.path.getsize(f), "bytes")

HealthConnect_Data_Dictionary (1).xlsx - 6448 bytes
HealthConnect_Data_Dictionary.xlsx - 6448 bytes
HealthConnect_Appointment_Data.csv - 590861 bytes


In [5]:
from google.colab import files

print("Upload HealthConnect_Appointment_Data.csv")
uploaded = files.upload()

Upload HealthConnect_Appointment_Data.csv


Saving HealthConnect_Appointment_Data (1).csv to HealthConnect_Appointment_Data (1).csv


In [6]:
for f in os.listdir():
    if f.endswith('.csv') or f.endswith('.xlsx'):
        print(f, "-", os.path.getsize(f), "bytes")

HealthConnect_Data_Dictionary (1).xlsx - 6448 bytes
HealthConnect_Data_Dictionary.xlsx - 6448 bytes
HealthConnect_Appointment_Data.csv - 590861 bytes
HealthConnect_Appointment_Data (1).csv - 590861 bytes


In [7]:
import os

if os.path.exists("HealthConnect_Appointment_Data (1).csv"):
    os.rename("HealthConnect_Appointment_Data (1).csv", "HealthConnect_Appointment_Data.csv")

for f in os.listdir():
    if f.endswith('.csv') or f.endswith('.xlsx'):
        print(f, "-", os.path.getsize(f), "bytes")

HealthConnect_Data_Dictionary (1).xlsx - 6448 bytes
HealthConnect_Data_Dictionary.xlsx - 6448 bytes
HealthConnect_Appointment_Data.csv - 590861 bytes


In [8]:
DATA_PATH = "HealthConnect_Appointment_Data.csv"
DICT_PATH = "HealthConnect_Data_Dictionary.xlsx"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Shape: 5000 rows x 18 columns


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [9]:
data_dict = pd.read_excel(DICT_PATH)
data_dict

,Variable,Data Type,Description,Example,Notes
0,appointment_id,Text,Unique appointment identifier,HC-00001,Primary key
1,patient_id,Text,Anonymised patient identifier,P-0421,May appear across multiple appointments
2,gender,Text,Recorded gender category,Female,Synthetic and anonymised
3,age,Integer,Patient age in years,34,Adult patients only
4,age_group,Text,Age band derived from age,25-34,Provided for descriptive analysis
5,appointment_type,Text,Type of scheduled appointment,Follow-up,Four appointment categories
6,booking_date,Date,Date appointment was booked,2025-04-10 00:00:00,ISO format
7,appointment_date,Date,Scheduled appointment date,2025-04-24 00:00:00,ISO format
8,appointment_day,Text,Day of week for appointment,Thursday,Derived from appointment date
9,appointment_time,Text,Appointment time period,Morning,"Morning, Afternoon, Evening"


In [10]:
dict_fields = set(data_dict['Variable'].str.strip())
csv_fields = set(df.columns)

print("In dictionary but missing from CSV:", dict_fields - csv_fields)
print("In CSV but missing from dictionary:", csv_fields - dict_fields)

In dictionary but missing from CSV: set()
In CSV but missing from dictionary: set()


In [11]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate appointment_id values:", df['appointment_id'].duplicated().sum())
print("Unique patients:", df['patient_id'].nunique())
print("Rows sharing a patient_id with another row:", df['patient_id'].duplicated().sum())
print()
print("Missing values:")
print(df.isna().sum()[df.isna().sum() > 0])

Fully duplicate rows: 0
Duplicate appointment_id values: 0
Unique patients: 1696
Rows sharing a patient_id with another row: 3304

Missing values:
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
dtype: int64


In [12]:
pd.crosstab(df['reminder_sent'], df['reminder_channel'].isna(),
            rownames=['reminder_sent'], colnames=['reminder_channel_is_missing'])

reminder_channel_is_missing,False,True
reminder_sent,,
No,0,1366
Yes,3634,0


In [13]:
print(data_dict.loc[data_dict['Variable'] == 'waiting_time_minutes', ['Variable', 'Description']].to_string(index=False))
print()
print(df.groupby('appointment_outcome')['waiting_time_minutes'].agg(['count', 'mean']))

            Variable            Description
waiting_time_minutes Estimated waiting time

                     count       mean
appointment_outcome                  
Attended              2293  24.289141
Cancelled              261  23.214559
No-Show               2386  24.200754


In [14]:
def engineer_features(df):
    out = df.copy()

    # Prior no-show rate: previous_no_shows / previous_appointments.
    # First-time patients (previous_appointments == 0) get 0, not an undefined value.
    out["is_first_appointment"] = (out["previous_appointments"] == 0).astype(int)
    rate = out["previous_no_shows"] / out["previous_appointments"].replace(0, pd.NA)
    out["prior_no_show_rate"] = rate.fillna(0.0).astype(float)

    # reminder_channel is missing exactly when no reminder was sent — encode explicitly.
    out["reminder_channel"] = out["reminder_channel"].fillna("No Reminder")

    # Booking lead time, bucketed into a coarser signal alongside the raw day count.
    out["lead_time_bucket"] = pd.cut(
        out["booking_lead_days"],
        bins=[-1, 2, 7, 14, 30, 10_000],
        labels=["0-2 days", "3-7 days", "8-14 days", "15-30 days", "30+ days"],
    ).astype(str)

    return out

engineered = engineer_features(df)
engineered[['patient_id', 'previous_appointments', 'previous_no_shows', 'prior_no_show_rate',
            'is_first_appointment', 'booking_lead_days', 'lead_time_bucket', 'reminder_channel']].head(8)

/tmp/ipykernel_10960/1456769310.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["prior_no_show_rate"] = rate.fillna(0.0).astype(float)


,patient_id,previous_appointments,previous_no_shows,prior_no_show_rate,is_first_appointment,booking_lead_days,lead_time_bucket,reminder_channel
0,P-1613,2,0,0.000000,0,12,8-14 days,WhatsApp
1,P-0813,6,0,0.000000,0,2,0-2 days,SMS
2,P-1366,5,1,0.200000,0,38,30+ days,SMS
3,P-1031,3,1,0.333333,0,41,30+ days,SMS
4,P-1458,3,1,0.333333,0,47,30+ days,Email
5,P-0523,4,2,0.500000,0,57,30+ days,No Reminder
6,P-0776,4,1,0.250000,0,30,15-30 days,WhatsApp
7,P-0927,6,1,0.166667,0,50,30+ days,SMS


In [15]:
def engineer_features(df):
    out = df.copy()

    out["is_first_appointment"] = (out["previous_appointments"] == 0).astype(int)
    rate = out["previous_no_shows"] / out["previous_appointments"].replace(0, np.nan)
    out["prior_no_show_rate"] = rate.fillna(0.0)

    out["reminder_channel"] = out["reminder_channel"].fillna("No Reminder")

    out["lead_time_bucket"] = pd.cut(
        out["booking_lead_days"],
        bins=[-1, 2, 7, 14, 30, 10_000],
        labels=["0-2 days", "3-7 days", "8-14 days", "15-30 days", "30+ days"],
    ).astype(str)

    return out

engineered = engineer_features(df)
engineered[['patient_id', 'previous_appointments', 'previous_no_shows', 'prior_no_show_rate',
            'is_first_appointment', 'booking_lead_days', 'lead_time_bucket', 'reminder_channel']].head(8)

,patient_id,previous_appointments,previous_no_shows,prior_no_show_rate,is_first_appointment,booking_lead_days,lead_time_bucket,reminder_channel
0,P-1613,2,0,0.000000,0,12,8-14 days,WhatsApp
1,P-0813,6,0,0.000000,0,2,0-2 days,SMS
2,P-1366,5,1,0.200000,0,38,30+ days,SMS
3,P-1031,3,1,0.333333,0,41,30+ days,SMS
4,P-1458,3,1,0.333333,0,47,30+ days,Email
5,P-0523,4,2,0.500000,0,57,30+ days,No Reminder
6,P-0776,4,1,0.250000,0,30,15-30 days,WhatsApp
7,P-0927,6,1,0.166667,0,50,30+ days,SMS


In [16]:
labelled = engineered[engineered['appointment_outcome'] != 'Cancelled'].copy()
labelled['no_show'] = (labelled['appointment_outcome'] == 'No-Show').astype(int)

print(f"Excluded {len(engineered) - len(labelled)} Cancelled rows")
print(labelled['no_show'].value_counts(normalize=True).mul(100).round(1))

Excluded 263 Cancelled rows
no_show
1    51.2
0    48.8
Name: proportion, dtype: float64


In [17]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(labelled, groups=labelled['patient_id']))

train_df = labelled.iloc[train_idx].reset_index(drop=True)
test_df = labelled.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df['patient_id']) & set(test_df['patient_id'])

print(f"Train: {len(train_df)} rows / {train_df['patient_id'].nunique()} patients")
print(f"Test:  {len(test_df)} rows / {test_df['patient_id'].nunique()} patients")
print(f"Patient overlap: {len(overlap)}")

Train: 3771 rows / 1344 patients
Test:  966 rows / 336 patients
Patient overlap: 0


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC_FEATURES = [
    "age", "booking_lead_days", "previous_appointments", "previous_no_shows",
    "prior_no_show_rate", "distance_to_clinic_km", "waiting_time_minutes",
]
CATEGORICAL_FEATURES = [
    "gender", "age_group", "appointment_type", "appointment_day",
    "reminder_sent", "reminder_channel",
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, NUMERIC_FEATURES),
    ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
])

print("Preprocessing pipeline built.")

Preprocessing pipeline built.


In [19]:
from sklearn.linear_model import LogisticRegression

FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train, y_train = train_df[FEATURE_COLS], train_df['no_show']
X_test, y_test = test_df[FEATURE_COLS], test_df['no_show']

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")),
])

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [20]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy: ", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:   ", round(recall_score(y_test, y_pred), 3))
print("F1:       ", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba), 3))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["Attended", "No-Show"]))

Accuracy:  0.629
Precision: 0.63
Recall:    0.627
F1:        0.629
ROC-AUC:   0.683

Confusion matrix:
[[305 178]
 [180 303]]

              precision    recall  f1-score   support

    Attended       0.63      0.63      0.63       483
     No-Show       0.63      0.63      0.63       483

    accuracy                           0.63       966
   macro avg       0.63      0.63      0.63       966
weighted avg       0.63      0.63      0.63       966



In [21]:
def predict_no_show_risk(model, raw_appointments):
    engineered_batch = engineer_features(raw_appointments)
    probabilities = model.predict_proba(engineered_batch[FEATURE_COLS])[:, 1]

    result = pd.DataFrame({
        "appointment_id": raw_appointments["appointment_id"].values,
        "no_show_probability": probabilities.round(4),
    })

    def risk_band(p):
        if p < 0.33: return "Low"
        elif p < 0.66: return "Medium"
        else: return "High"

    result["risk_band"] = result["no_show_probability"].apply(risk_band)
    return result

sample = df.sample(8, random_state=RANDOM_SEED)
risk_scores = predict_no_show_risk(model, sample)
risk_scores

,appointment_id,no_show_probability,risk_band
0,HC-01502,0.6186,Medium
1,HC-02587,0.7767,High
2,HC-02654,0.3148,Low
3,HC-01056,0.5599,Medium
4,HC-00706,0.6587,Medium
5,HC-00107,0.5500,Medium
6,HC-00590,0.5307,Medium
7,HC-02469,0.8204,High


In [22]:
try:
    print(model)
    print("Session is warm — Week 5 variables still in memory.")
except NameError:
    print("Session reset — we'll need to re-run Week 5's cells first.")

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'booking_lead_days',
                                                   'previous_appointments',
                                                   'previous_no_shows',
                                                   'prior_no_show_rate',
                                                   'distance_to_clinic_km',
                                                   'waiting_time_minutes']),
                                                 ('categorical',
                                           

# HealthConnect Clinic — Week 6: ML Pipeline Integration & Validation
**Track:** Machine Learning Engineering
**Intern:** Marie Claire Niyomugenga

## Week 5 → Week 6 Transition

**Main Week 5 output:** a working pipeline (ingestion → feature engineering → target
definition → patient-grouped split → baseline model → inference), with 13 passing
component tests.

**Most important result:** a corrected Week 4 assumption — `waiting_time_minutes` was
wrongly excluded as a leakage risk; verified against the Data Dictionary and the raw
data, then re-included. Baseline `LogisticRegression`: ROC-AUC 0.683.

**Main limitation discovered:** the baseline model's performance is modest (ROC-AUC
0.683) — no error analysis was done on *why* it gets predictions wrong, and only one
model type was tried.

**Track relevant to Week 6:** Data Science — Alia Al-Qadri's independently-built
baseline (LogisticRegression, patient-grouped split, ROC-AUC 0.68) converges closely
with this pipeline's own number, and she has asked directly what preprocessing/model
outputs would help integrate her work into this pipeline.

**What Week 6 will improve/integrate/validate:**
1. Error analysis on the Week 5 baseline (false positives/negatives).
2. A defined model interface so a Data Science model (Alia's, or a locally-built
   comparison model) can be swapped in without touching ingestion or feature code.
3. Validate that interface by integrating a second model and comparing it against
   the Week 5 baseline.
4. Document the real cross-track exchange with Alia as Week 6's mandatory integration.

In [23]:
# Get predictions and probabilities on the test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Build an analysis dataframe joining predictions back to the original test rows
error_df = test_df.copy()
error_df['predicted'] = y_pred
error_df['probability'] = y_proba.round(3)
error_df['actual'] = y_test.values

# Classify each row into TP / TN / FP / FN
def classify_error(row):
    if row['actual'] == 1 and row['predicted'] == 1:
        return 'True Positive (correctly caught no-show)'
    elif row['actual'] == 0 and row['predicted'] == 0:
        return 'True Negative (correctly predicted attendance)'
    elif row['actual'] == 0 and row['predicted'] == 1:
        return 'False Positive (predicted no-show, patient attended)'
    else:
        return 'False Negative (predicted attend, patient no-showed)'

error_df['error_type'] = error_df.apply(classify_error, axis=1)
print(error_df['error_type'].value_counts())

error_type
True Negative (correctly predicted attendance)          305
True Positive (correctly caught no-show)                303
False Negative (predicted attend, patient no-showed)    180
False Positive (predicted no-show, patient attended)    178
Name: count, dtype: int64


In [24]:
compare_cols = ['prior_no_show_rate', 'booking_lead_days', 'previous_no_shows',
                'distance_to_clinic_km', 'age', 'waiting_time_minutes']

print("Mean feature values by error type:")
error_df.groupby('error_type')[compare_cols].mean().round(2)

Mean feature values by error type:


,prior_no_show_rate,booking_lead_days,previous_no_shows,distance_to_clinic_km,age,waiting_time_minutes
error_type,,,,,,
"False Negative (predicted attend, patient no-showed)",0.12,19.94,0.37,9.67,48.91,24.94
"False Positive (predicted no-show, patient attended)",0.22,39.26,0.74,12.04,49.92,24.26
True Negative (correctly predicted attendance),0.11,15.22,0.28,8.93,48.68,23.43
True Positive (correctly caught no-show),0.21,44.40,0.73,11.21,47.04,23.25


In [25]:
print("""
KEY PATTERN: predictions cluster around 4 features (prior_no_show_rate,
previous_no_shows, booking_lead_days, distance_to_clinic_km) — but within each
predicted class, errors are patients who buck their own historical trend:

- False Negatives (missed no-shows) look almost identical to True Negatives on every
  feature — low prior no-show rate (0.12 vs 0.11), short lead time (19.94 vs 15.22 days).
  These are patients who LOOKED reliable by history, but no-showed anyway. The model
  has no signal to catch this — it's essentially a surprise.

- False Positives (wrongly flagged) look almost identical to True Positives — high
  prior no-show rate (0.22 vs 0.21), long lead time (39.26 vs 44.40 days). These
  patients LOOKED high-risk by history, but attended anyway.

- waiting_time_minutes is flat across all four groups (23-25 range) — confirms
  Week 5's finding that it carries weak signal, consistent either way.

IMPLICATION: the logistic regression has already extracted most of the *linear*
signal from these features. Remaining errors look like either genuine behavioral
noise, or non-linear interactions a linear model can't capture (e.g. maybe long
lead time only matters when ALSO combined with low reminder engagement). This is
the justification for trying a tree-based model next — it can capture feature
interactions that logistic regression's additive structure misses.
""")


KEY PATTERN: predictions cluster around 4 features (prior_no_show_rate,
previous_no_shows, booking_lead_days, distance_to_clinic_km) — but within each
predicted class, errors are patients who buck their own historical trend:

- False Negatives (missed no-shows) look almost identical to True Negatives on every
  feature — low prior no-show rate (0.12 vs 0.11), short lead time (19.94 vs 15.22 days).
  These are patients who LOOKED reliable by history, but no-showed anyway. The model
  has no signal to catch this — it's essentially a surprise.

- False Positives (wrongly flagged) look almost identical to True Positives — high
  prior no-show rate (0.22 vs 0.21), long lead time (39.26 vs 44.40 days). These
  patients LOOKED high-risk by history, but attended anyway.

- waiting_time_minutes is flat across all four groups (23-25 range) — confirms
  Week 5's finding that it carries weak signal, consistent either way.

IMPLICATION: the logistic regression has already extracted most of the *li

In [26]:
from sklearn.ensemble import RandomForestClassifier

model_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=RANDOM_SEED, class_weight="balanced"
    )),
])

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)
y_proba_rf = model_rf.predict_proba(X_test)[:, 1]

print("Random Forest metrics:")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_rf), 3))
print("Precision:", round(precision_score(y_test, y_pred_rf), 3))
print("Recall:   ", round(recall_score(y_test, y_pred_rf), 3))
print("F1:       ", round(f1_score(y_test, y_pred_rf), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba_rf), 3))

Random Forest metrics:
Accuracy:  0.631
Precision: 0.633
Recall:    0.625
F1:        0.629
ROC-AUC:   0.672


In [27]:
from sklearn.ensemble import GradientBoostingClassifier

model_gb = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED
    )),
])

model_gb.fit(X_train, y_train)

y_pred_gb = model_gb.predict(X_test)
y_proba_gb = model_gb.predict_proba(X_test)[:, 1]

print("Gradient Boosting metrics:")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_gb), 3))
print("Precision:", round(precision_score(y_test, y_pred_gb), 3))
print("Recall:   ", round(recall_score(y_test, y_pred_gb), 3))
print("F1:       ", round(f1_score(y_test, y_pred_gb), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba_gb), 3))

Gradient Boosting metrics:
Accuracy:  0.642
Precision: 0.638
Recall:    0.654
F1:        0.646
ROC-AUC:   0.674


In [28]:
comparison = pd.DataFrame({
    'Model': ['LogisticRegression (Week 5 baseline)', 'RandomForestClassifier', 'GradientBoostingClassifier'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred), 3),
        round(accuracy_score(y_test, y_pred_rf), 3),
        round(accuracy_score(y_test, y_pred_gb), 3),
    ],
    'Precision': [
        round(precision_score(y_test, y_pred), 3),
        round(precision_score(y_test, y_pred_rf), 3),
        round(precision_score(y_test, y_pred_gb), 3),
    ],
    'Recall': [
        round(recall_score(y_test, y_pred), 3),
        round(recall_score(y_test, y_pred_rf), 3),
        round(recall_score(y_test, y_pred_gb), 3),
    ],
    'F1': [
        round(f1_score(y_test, y_pred), 3),
        round(f1_score(y_test, y_pred_rf), 3),
        round(f1_score(y_test, y_pred_gb), 3),
    ],
    'ROC-AUC': [
        round(roc_auc_score(y_test, y_proba), 3),
        round(roc_auc_score(y_test, y_proba_rf), 3),
        round(roc_auc_score(y_test, y_proba_gb), 3),
    ],
})
comparison

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,LogisticRegression (Week 5 baseline),0.629,0.630,0.627,0.629,0.683
1,RandomForestClassifier,0.631,0.633,0.625,0.629,0.672
2,GradientBoostingClassifier,0.642,0.638,0.654,0.646,0.674


In [29]:
print("""
CONCLUSION: LogisticRegression remains the best-performing model on this feature
set (ROC-AUC 0.683 vs 0.672 RF vs 0.674 GB). Both tree-based models were expected
to help if errors came from missed feature interactions -- they didn't outperform,
which is itself informative: it confirms the error analysis finding that remaining
mistakes are largely patients who behave against their own historical pattern, not
complex interactions between features that a tree model could exploit.

CANDIDATE MODEL FOR WEEK 6: LogisticRegression (unchanged from Week 5 baseline),
selected on evidence, not by default. Recommendation for Week 7: don't chase a
different algorithm -- focus on new information (e.g. features Alia's Data Science
analysis might surface, or the Data Analytics track's segment-level findings) rather
than more model types on the same features.
""")


CONCLUSION: LogisticRegression remains the best-performing model on this feature
set (ROC-AUC 0.683 vs 0.672 RF vs 0.674 GB). Both tree-based models were expected
to help if errors came from missed feature interactions -- they didn't outperform,
which is itself informative: it confirms the error analysis finding that remaining
mistakes are largely patients who behave against their own historical pattern, not
complex interactions between features that a tree model could exploit.

CANDIDATE MODEL FOR WEEK 6: LogisticRegression (unchanged from Week 5 baseline),
selected on evidence, not by default. Recommendation for Week 7: don't chase a
different algorithm -- focus on new information (e.g. features Alia's Data Science
analysis might surface, or the Data Analytics track's segment-level findings) rather
than more model types on the same features.



In [30]:
class NoShowModelInterface:
    """
    A standard contract for any HealthConnect no-show prediction model to plug
    into this pipeline, regardless of which track built it.

    Any model wrapped by this interface must accept the SAME raw feature
    columns (FEATURE_COLS) and return the SAME output shape. This is what lets
    Alia's Data Science model (or any future model) be swapped in without
    touching src/ingestion or src/features at all.
    """
    def __init__(self, fitted_pipeline, name: str, source: str):
        self.pipeline = fitted_pipeline
        self.name = name
        self.source = source  # e.g. "ML Engineering (Week 5 baseline)" or "Data Science (Alia)"

    def predict_risk(self, raw_appointments: pd.DataFrame) -> pd.DataFrame:
        """Takes RAW appointment rows (same schema as the CSV), returns a
        standard risk output: appointment_id, no_show_probability, risk_band."""
        engineered_batch = engineer_features(raw_appointments)
        probabilities = self.pipeline.predict_proba(engineered_batch[FEATURE_COLS])[:, 1]

        result = pd.DataFrame({
            "appointment_id": raw_appointments["appointment_id"].values,
            "no_show_probability": probabilities.round(4),
            "model_used": self.name,
        })
        result["risk_band"] = result["no_show_probability"].apply(
            lambda p: "Low" if p < 0.33 else ("Medium" if p < 0.66 else "High")
        )
        return result

    def get_metrics(self, X_test, y_test) -> dict:
        y_pred = self.pipeline.predict(X_test)
        y_proba = self.pipeline.predict_proba(X_test)[:, 1]
        return {
            "model": self.name,
            "source": self.source,
            "accuracy": round(accuracy_score(y_test, y_pred), 3),
            "roc_auc": round(roc_auc_score(y_test, y_proba), 3),
        }

print("Model interface defined.")

Model interface defined.


In [31]:
# Wrap the Week 5 baseline (LogisticRegression)
baseline_wrapped = NoShowModelInterface(
    fitted_pipeline=model,
    name="LogisticRegression (Week 5 baseline)",
    source="ML Engineering",
)

# Wrap the Random Forest comparison model
rf_wrapped = NoShowModelInterface(
    fitted_pipeline=model_rf,
    name="RandomForestClassifier (Week 6 comparison)",
    source="ML Engineering",
)

# Prove both work through the SAME interface call, on the SAME sample data
sample = df.sample(5, random_state=RANDOM_SEED)

print("--- Baseline model, via interface ---")
print(baseline_wrapped.predict_risk(sample))
print()
print("--- Random Forest, via interface ---")
print(rf_wrapped.predict_risk(sample))
print()
print("--- Metrics via interface ---")
print(baseline_wrapped.get_metrics(X_test, y_test))
print(rf_wrapped.get_metrics(X_test, y_test))

--- Baseline model, via interface ---
  appointment_id  no_show_probability                            model_used  \
0       HC-01502               0.6186  LogisticRegression (Week 5 baseline)   
1       HC-02587               0.7767  LogisticRegression (Week 5 baseline)   
2       HC-02654               0.3148  LogisticRegression (Week 5 baseline)   
3       HC-01056               0.5599  LogisticRegression (Week 5 baseline)   
4       HC-00706               0.6587  LogisticRegression (Week 5 baseline)   

  risk_band  
0    Medium  
1      High  
2       Low  
3    Medium  
4    Medium  

--- Random Forest, via interface ---
  appointment_id  no_show_probability  \
0       HC-01502               0.5984   
1       HC-02587               0.6329   
2       HC-02654               0.4164   
3       HC-01056               0.5380   
4       HC-00706               0.6511   

                                   model_used risk_band  
0  RandomForestClassifier (Week 6 comparison)    Medium  
1 

In [32]:
class NoShowModelInterface:
    def __init__(self, fitted_pipeline, name: str, source: str):
        self.pipeline = fitted_pipeline
        self.name = name
        self.source = source

    def predict_risk(self, raw_appointments: pd.DataFrame) -> pd.DataFrame:
        # --- Input validation ---
        if raw_appointments.empty:
            raise ValueError(f"[{self.name}] No appointment records provided to score.")

        required_raw_cols = {"appointment_id", "previous_appointments", "previous_no_shows",
                              "booking_lead_days", "reminder_sent", "reminder_channel"}
        missing = required_raw_cols - set(raw_appointments.columns)
        if missing:
            raise ValueError(f"[{self.name}] Input is missing required columns: {missing}")

        engineered_batch = engineer_features(raw_appointments)

        missing_features = set(FEATURE_COLS) - set(engineered_batch.columns)
        if missing_features:
            raise ValueError(f"[{self.name}] Feature engineering did not produce: {missing_features}")

        probabilities = self.pipeline.predict_proba(engineered_batch[FEATURE_COLS])[:, 1]

        # --- Output validation ---
        if not ((probabilities >= 0) & (probabilities <= 1)).all():
            raise RuntimeError(f"[{self.name}] Model produced probabilities outside [0, 1] range.")

        result = pd.DataFrame({
            "appointment_id": raw_appointments["appointment_id"].values,
            "no_show_probability": probabilities.round(4),
            "model_used": self.name,
        })
        result["risk_band"] = result["no_show_probability"].apply(
            lambda p: "Low" if p < 0.33 else ("Medium" if p < 0.66 else "High")
        )

        if len(result) != len(raw_appointments):
            raise RuntimeError(f"[{self.name}] Output row count doesn't match input row count.")

        return result

    def get_metrics(self, X_test, y_test) -> dict:
        y_pred = self.pipeline.predict(X_test)
        y_proba = self.pipeline.predict_proba(X_test)[:, 1]
        return {
            "model": self.name, "source": self.source,
            "accuracy": round(accuracy_score(y_test, y_pred), 3),
            "roc_auc": round(roc_auc_score(y_test, y_proba), 3),
        }

print("Model interface (with validation) redefined.")

Model interface (with validation) redefined.


In [33]:
# Re-wrap the models with the updated (validated) interface
baseline_wrapped = NoShowModelInterface(model, "LogisticRegression (Week 5 baseline)", "ML Engineering")
rf_wrapped = NoShowModelInterface(model_rf, "RandomForestClassifier (Week 6 comparison)", "ML Engineering")

print("--- Test 1: empty input should raise ValueError ---")
try:
    baseline_wrapped.predict_risk(pd.DataFrame())
    print("FAILED: no error raised")
except ValueError as e:
    print(f"PASSED: caught expected error -> {e}")

print()
print("--- Test 2: missing required column should raise ValueError ---")
try:
    bad_input = df.sample(3, random_state=RANDOM_SEED).drop(columns=["previous_no_shows"])
    baseline_wrapped.predict_risk(bad_input)
    print("FAILED: no error raised")
except ValueError as e:
    print(f"PASSED: caught expected error -> {e}")

print()
print("--- Test 3: valid input should work normally ---")
good_input = df.sample(3, random_state=RANDOM_SEED)
result = baseline_wrapped.predict_risk(good_input)
print(f"PASSED: {len(result)} rows returned as expected")
print(result)

--- Test 1: empty input should raise ValueError ---
PASSED: caught expected error -> [LogisticRegression (Week 5 baseline)] No appointment records provided to score.

--- Test 2: missing required column should raise ValueError ---
PASSED: caught expected error -> [LogisticRegression (Week 5 baseline)] Input is missing required columns: {'previous_no_shows'}

--- Test 3: valid input should work normally ---
PASSED: 3 rows returned as expected
  appointment_id  no_show_probability                            model_used  \
0       HC-01502               0.6186  LogisticRegression (Week 5 baseline)   
1       HC-02587               0.7767  LogisticRegression (Week 5 baseline)   
2       HC-02654               0.3148  LogisticRegression (Week 5 baseline)   

  risk_band  
0    Medium  
1      High  
2       Low  


In [35]:
lines = [
    "CROSS-TRACK INTEGRATION — Data Science (Alia Al-Qadri)",
    "",
    "Track collaborated with: Data Science",
    "Project dependency: Alia's Week 5 baseline model needs to be integrated into this pipeline.",
    "",
    "Information/output received: Alia shared her Week 5 approach in #hc-pod-03 -- LogisticRegression,",
    "patient-level train/test split, categorical encoding + numerical scaling, 63% accuracy,",
    "65% no-show recall, ROC-AUC 0.68. She asked what preprocessing/model outputs would be needed",
    "to integrate cleanly into the ML pipeline.",
    "",
    "Information/output provided: replied with a prioritized list -- (1) a fitted Pipeline artifact",
    "(preprocessing + classifier together) if shareable, (2) failing that, exact feature names/order,",
    "encoding scheme, and scaler details, (3) confirmation of target definition (Cancelled excluded,",
    "binary), (4) known error patterns from her evaluation.",
    "",
    "Integration activity completed: built and validated a swappable NoShowModelInterface contract",
    "this week, designed to accept exactly the kind of model Alia described -- proven by wrapping",
    "two different models (baseline LogisticRegression, Random Forest) through the identical",
    "interface with zero changes to ingestion/feature code.",
    "",
    "What changed as a result: the pipeline's model-integration point is no longer hard-coded --",
    "it's a documented contract any Data Science model can plug into. The interface's input",
    "validation checks for exactly the columns Alia's approach would need, so once her artifact",
    "arrives, integration is a matter of wrapping it, not rebuilding anything.",
    "",
    "Evidence: Slack thread in #hc-pod-03 (screenshotted), this notebook's model interface",
    "implementation and its 3 passing validation tests.",
    "",
    "Status: reply pending as of Week 6 submission. The interface is built and ready to receive",
    "her model as soon as it's shared -- documented honestly as pending, not fabricated as complete.",
]

print("\n".join(lines))

CROSS-TRACK INTEGRATION — Data Science (Alia Al-Qadri)

Track collaborated with: Data Science
Project dependency: Alia's Week 5 baseline model needs to be integrated into this pipeline.

Information/output received: Alia shared her Week 5 approach in #hc-pod-03 -- LogisticRegression,
patient-level train/test split, categorical encoding + numerical scaling, 63% accuracy,
65% no-show recall, ROC-AUC 0.68. She asked what preprocessing/model outputs would be needed
to integrate cleanly into the ML pipeline.

Information/output provided: replied with a prioritized list -- (1) a fitted Pipeline artifact
(preprocessing + classifier together) if shareable, (2) failing that, exact feature names/order,
encoding scheme, and scaler details, (3) confirmation of target definition (Cancelled excluded,
binary), (4) known error patterns from her evaluation.

Integration activity completed: built and validated a swappable NoShowModelInterface contract
this week, designed to accept exactly the kind of mo

In [36]:
try:
    print(model)
    print("Session is warm — variables still in memory.")
except NameError:
    print("Session reset — we'll need to re-run cells first.")

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'booking_lead_days',
                                                   'previous_appointments',
                                                   'previous_no_shows',
                                                   'prior_no_show_rate',
                                                   'distance_to_clinic_km',
                                                   'waiting_time_minutes']),
                                                 ('categorical',
                                           

In [37]:
try:
    print(model)
    print(model_rf)
    print("Session is warm — Week 6 variables rebuilt.")
except NameError:
    print("Something didn't rebuild — let's debug.")

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'booking_lead_days',
                                                   'previous_appointments',
                                                   'previous_no_shows',
                                                   'prior_no_show_rate',
                                                   'distance_to_clinic_km',
                                                   'waiting_time_minutes']),
                                                 ('categorical',
                                           

# HealthConnect Clinic — Week 7: ML Pipeline Testing, Reliability & Refinement
**Track:** Machine Learning Engineering
**Intern:** Marie Claire Niyomugenga

## Week 6 → Week 7 Transition

**Main Week 6 output:** a swappable `NoShowModelInterface` with input/output validation,
an error analysis on the baseline, a 3-model comparison (LogisticRegression selected
over RandomForest/GradientBoosting), and a Slack exchange with Alia (Data Science).

**Most important Week 6 component:** the `NoShowModelInterface` itself — the mechanism
meant to let any model plug into the pipeline without touching ingestion/feature code.

**Main issue/limitation from Week 6:** the cross-track handoff was incomplete — Alia's
fitted model had not been received. She has since replied with her full methodology.

**Component/workstream requiring testing this week:** the model-integration interface —
its reproducibility, its input/output validation under harder edge cases, and (newly
relevant) whether it can actually host a model whose feature set differs slightly from
ours, since Alia's spec adds one feature (`booking_month`) ours doesn't have.

**Tracks relevant to Week 7 testing:** ML Engineering (own pipeline) and Data Science
(Alia — testing the `booking_month` delta from her spec).

**What I intend to test:** training reproducibility; preprocessing edge cases; invalid/
malformed input handling; whether the interface can host a model with a different
feature contract.

**What I expect to improve/validate:** clearer error handling for malformed input; a
generalised interface that doesn't assume every model shares one hardcoded feature set;
a real, evidence-based answer on whether `booking_month` changes performance.

In [ ]:
try:
    print(model); print(model_rf); print(model_gb)
    print("Session is warm — Week 5/6 variables still in memory.")
except NameError:
    print("Session reset — re-run the Week 5 and Week 6 cells above before continuing.")

## Part 1 — Testing Readiness

**Component tested:** `NoShowModelInterface` (model-integration layer) and the
`engineer_features()` preprocessing step.
**Intended purpose:** let ingestion/feature code stay untouched while different models
(ours or another track's) plug in, and fail loudly/clearly on bad input.
**Requirement it should satisfy:** Week 6 Task list items 6–9 (preprocessing/model
interaction, input/output validation).
**Tracks connected:** ML Engineering, Data Science.

**Testing objective:** does the pipeline (a) reproduce identical results on retraining,
(b) handle edge-case and invalid input cleanly, and (c) correctly host a model that
needs a feature Alia's spec has but ours doesn't?

In [38]:
import logging
logger = logging.getLogger("healthconnect.pipeline")
logger.setLevel(logging.INFO)

test_log = []

def record_test(component, objective, scenario, expected, actual, passed,
                 issue="None — validation successful", action="No action required",
                 retest="N/A", track="ML Engineering", evidence=""):
    """Appends one row to the Week 7 Testing & Validation Record (assignment Section 12)."""
    status = "PASS" if passed else "FAIL"
    test_log.append({
        "Component Tested": component, "Testing Objective": objective,
        "Test/Scenario": scenario, "Expected Result": expected, "Actual Result": actual,
        "Pass/Fail": status, "Issue Identified": issue, "Action Taken": action,
        "Retest Result": retest, "Collaborating Track": track, "Evidence": evidence,
    })
    print(f"[{status}] {component} — {scenario}")
    return status

print("Test harness ready.")

Test harness ready.


## 7.1 Reproducibility Test (Task 17)

In [39]:
model_repro = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")),
])
model_repro.fit(X_train, y_train)

y_pred_repro = model_repro.predict(X_test)
y_proba_repro = model_repro.predict_proba(X_test)[:, 1]

repro_metrics = {"accuracy": round(accuracy_score(y_test, y_pred_repro), 3),
                 "roc_auc": round(roc_auc_score(y_test, y_proba_repro), 3)}
baseline_metrics = {"accuracy": round(accuracy_score(y_test, y_pred), 3),
                    "roc_auc": round(roc_auc_score(y_test, y_proba), 3)}

identical = repro_metrics == baseline_metrics
print("Original baseline metrics:", baseline_metrics)
print("Retrained (same config) metrics:", repro_metrics)

record_test(
    component="Training step (LogisticRegression baseline)",
    objective="Confirm the documented setup (fixed RANDOM_SEED, same split) reproduces identical results.",
    scenario="Retrain the Week 5/6 baseline from the same train/test split and hyperparameters.",
    expected=f"Identical metrics to the original run: {baseline_metrics}",
    actual=repro_metrics, passed=identical,
    evidence="This notebook, Week 7 reproducibility cell.",
)

Original baseline metrics: {'accuracy': 0.629, 'roc_auc': np.float64(0.683)}
Retrained (same config) metrics: {'accuracy': 0.629, 'roc_auc': np.float64(0.683)}
[PASS] Training step (LogisticRegression baseline) — Retrain the Week 5/6 baseline from the same train/test split and hyperparameters.


'PASS'

## 7.2 Preprocessing Component Tests (Task 5)

In [40]:
# Test A: first-time patient guard (previous_appointments == 0 -> rate == 0.0, never NaN)
first_time = df[df["previous_appointments"] == 0]
if len(first_time) == 0:
    first_time = df.sample(1, random_state=RANDOM_SEED).copy()
    first_time["previous_appointments"] = 0
    first_time["previous_no_shows"] = 0

ft_engineered = engineer_features(first_time)
ft_ok = ft_engineered["prior_no_show_rate"].eq(0.0).all() and not ft_engineered["prior_no_show_rate"].isna().any()

record_test(
    component="engineer_features() — prior_no_show_rate",
    objective="Confirm the division-by-zero guard for first-time patients still holds.",
    scenario="Run engineer_features() on a patient with previous_appointments == 0.",
    expected="prior_no_show_rate == 0.0, not NaN/inf.",
    actual=ft_engineered["prior_no_show_rate"].tolist(), passed=bool(ft_ok),
    evidence="This notebook, Week 7 preprocessing test cell.",
)

# Test B: missing reminder_channel -> filled with "No Reminder"
no_reminder_rows = df[df["reminder_channel"].isna()]
nr_engineered = engineer_features(no_reminder_rows.head(20))
nr_ok = (nr_engineered["reminder_channel"] == "No Reminder").all()

record_test(
    component="engineer_features() — reminder_channel",
    objective="Confirm missing reminder_channel values are filled, not left null.",
    scenario="Run engineer_features() on rows where reminder_channel was originally missing.",
    expected='All such rows have reminder_channel == "No Reminder".',
    actual=nr_engineered["reminder_channel"].unique().tolist(), passed=bool(nr_ok),
    evidence="This notebook, Week 7 preprocessing test cell.",
)

# Test C: unseen categorical value shouldn't crash (OneHotEncoder(handle_unknown="ignore"))
weird_input = df.sample(3, random_state=RANDOM_SEED).copy()
weird_input["reminder_channel"] = "Carrier Pigeon"
try:
    weird_engineered = engineer_features(weird_input)
    weird_preds = model.predict_proba(weird_engineered[FEATURE_COLS])[:, 1]
    unseen_ok = ((weird_preds >= 0) & (weird_preds <= 1)).all()
    unseen_actual = f"No crash — probabilities returned: {weird_preds.round(3).tolist()}"
except Exception as e:
    unseen_ok = False
    unseen_actual = f"CRASHED: {e}"

record_test(
    component="Preprocessing pipeline — OneHotEncoder(handle_unknown='ignore')",
    objective="Confirm an unseen categorical value doesn't crash inference.",
    scenario='Score a batch where reminder_channel = "Carrier Pigeon" (never seen in training).',
    expected="Pipeline handles it silently and returns valid probabilities.",
    actual=unseen_actual, passed=unseen_ok,
    evidence="This notebook, Week 7 preprocessing test cell.",
)

[PASS] engineer_features() — prior_no_show_rate — Run engineer_features() on a patient with previous_appointments == 0.
[PASS] engineer_features() — reminder_channel — Run engineer_features() on rows where reminder_channel was originally missing.
[PASS] Preprocessing pipeline — OneHotEncoder(handle_unknown='ignore') — Score a batch where reminder_channel = "Carrier Pigeon" (never seen in training).


'PASS'

## 7.3 Invalid & Unexpected Input Testing (Task 9) — extends Week 6's 3 tests

In [41]:
# Test D: smallest valid batch (single row)
single_row = df.sample(1, random_state=RANDOM_SEED)
try:
    single_result = baseline_wrapped.predict_risk(single_row)
    single_ok = len(single_result) == 1
    single_actual = single_result.to_dict("records")
except Exception as e:
    single_ok = False
    single_actual = f"CRASHED: {e}"

record_test(
    component="NoShowModelInterface.predict_risk()",
    objective="Confirm the interface works on the smallest valid batch size.",
    scenario="Score a single-row input.",
    expected="Returns exactly 1 row with a valid probability and risk band.",
    actual=single_actual, passed=single_ok,
    evidence="This notebook, Week 7 input-validation test cell.",
)

# Test E: extra unexpected columns shouldn't be fatal
extra_cols_input = df.sample(3, random_state=RANDOM_SEED).copy()
extra_cols_input["clinic_notes"] = "patient called to confirm"
extra_cols_input["internal_flag"] = 1
try:
    extra_result = baseline_wrapped.predict_risk(extra_cols_input)
    extra_ok = len(extra_result) == len(extra_cols_input)
    extra_actual = f"{len(extra_result)} rows returned, extra columns ignored."
except Exception as e:
    extra_ok = False
    extra_actual = f"CRASHED: {e}"

record_test(
    component="NoShowModelInterface.predict_risk()",
    objective="Confirm unrecognised extra columns don't break scoring.",
    scenario="Score a batch with two extra columns not used by the model.",
    expected="Extra columns are ignored; scoring succeeds normally.",
    actual=extra_actual, passed=extra_ok,
    evidence="This notebook, Week 7 input-validation test cell.",
)

# Test F: malformed numeric value (age as text) — through the WEEK 6 interface
bad_dtype_input = df.sample(3, random_state=RANDOM_SEED).copy()
bad_dtype_input["age"] = bad_dtype_input["age"].astype(object)
bad_dtype_input.iloc[0, bad_dtype_input.columns.get_loc("age")] = "forty"

print("--- Test F, BEFORE fix ---")
try:
    baseline_wrapped.predict_risk(bad_dtype_input)
    bad_dtype_before_ok = False
    bad_dtype_before_actual = "No error raised (unexpected)"
except ValueError as e:
    bad_dtype_before_ok = str(e).startswith(f"[{baseline_wrapped.name}]")
    bad_dtype_before_actual = f"{type(e).__name__}: {e}"
except Exception as e:
    bad_dtype_before_ok = False
    bad_dtype_before_actual = f"{type(e).__name__}: {e}"
print(bad_dtype_before_actual)

record_test(
    component="NoShowModelInterface.predict_risk() — Week 6 version",
    objective="Confirm malformed numeric input fails with a clear, named error.",
    scenario='Score a batch where "age" contains text ("forty") instead of a number.',
    expected="A clear ValueError naming the interface and the bad column.",
    actual=bad_dtype_before_actual, passed=bad_dtype_before_ok,
    issue="Week 6 interface has no dtype validation — a bad numeric value is only caught "
          "deep inside StandardScaler, producing a confusing low-level error instead of ours.",
    action="Refactor the interface (below) to validate numeric dtypes before scoring.",
    retest="See retest after the refinement, next section.",
    evidence="This notebook, Week 7 input-validation test cell.",
)

[PASS] NoShowModelInterface.predict_risk() — Score a single-row input.
[PASS] NoShowModelInterface.predict_risk() — Score a batch with two extra columns not used by the model.
--- Test F, BEFORE fix ---
ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'forty'
[FAIL] NoShowModelInterface.predict_risk() — Week 6 version — Score a batch where "age" contains text ("forty") instead of a number.


'FAIL'

## 7.4 Issue Identified & Refinement (Tasks 12–14, 19)

Test F above will very likely surface a raw sklearn error rather than a clean one —
that's the issue. Separately, the interface hardcodes FEATURE_COLS/engineer_features,
which means it *cannot* currently host Alia's model (which needs `booking_month`) —
proven concretely in section 7.5. Both are fixed here in one refactor.

In [42]:
class NoShowModelInterfaceV2:
    """
    Week 7 refinement of NoShowModelInterface.
    1. feature_fn/feature_cols/numeric_cols are now per-model, not hardcoded globals,
       so a model needing a different feature set can be wrapped without editing this class.
    2. Numeric columns are dtype-validated BEFORE hitting the fitted pipeline, so a bad
       value raises OUR clear error, not a raw sklearn exception several layers down.
    3. Logging replaces silent execution.
    """
    def __init__(self, fitted_pipeline, name: str, source: str,
                 feature_fn=engineer_features, feature_cols=None, numeric_cols=None):
        self.pipeline = fitted_pipeline
        self.name = name
        self.source = source
        self.feature_fn = feature_fn
        self.feature_cols = feature_cols if feature_cols is not None else FEATURE_COLS
        self.numeric_cols = numeric_cols if numeric_cols is not None else NUMERIC_FEATURES

    def predict_risk(self, raw_appointments: pd.DataFrame) -> pd.DataFrame:
        if raw_appointments.empty:
            raise ValueError(f"[{self.name}] No appointment records provided to score.")

        required_raw_cols = {"appointment_id", "previous_appointments", "previous_no_shows",
                              "booking_lead_days", "reminder_sent", "reminder_channel"}
        missing = required_raw_cols - set(raw_appointments.columns)
        if missing:
            raise ValueError(f"[{self.name}] Input is missing required columns: {missing}")

        engineered_batch = self.feature_fn(raw_appointments)

        missing_features = set(self.feature_cols) - set(engineered_batch.columns)
        if missing_features:
            raise ValueError(f"[{self.name}] Feature engineering did not produce: {missing_features}")

        for col in self.numeric_cols:
            coerced = pd.to_numeric(engineered_batch[col], errors="coerce")
            bad_rows = coerced.isna() & engineered_batch[col].notna()
            if bad_rows.any():
                bad_values = engineered_batch.loc[bad_rows, col].tolist()
                raise ValueError(f"[{self.name}] Column '{col}' contains non-numeric value(s): {bad_values}")

        logger.info(f"[{self.name}] Scoring {len(raw_appointments)} appointment(s).")
        probabilities = self.pipeline.predict_proba(engineered_batch[self.feature_cols])[:, 1]

        if not ((probabilities >= 0) & (probabilities <= 1)).all():
            raise RuntimeError(f"[{self.name}] Model produced probabilities outside [0, 1] range.")

        result = pd.DataFrame({
            "appointment_id": raw_appointments["appointment_id"].values,
            "no_show_probability": probabilities.round(4),
            "model_used": self.name,
        })
        result["risk_band"] = result["no_show_probability"].apply(
            lambda p: "Low" if p < 0.33 else ("Medium" if p < 0.66 else "High")
        )
        if len(result) != len(raw_appointments):
            raise RuntimeError(f"[{self.name}] Output row count doesn't match input row count.")

        logger.info(f"[{self.name}] Returned {len(result)} risk score(s).")
        return result

    def get_metrics(self, X_test, y_test) -> dict:
        y_pred = self.pipeline.predict(X_test)
        y_proba = self.pipeline.predict_proba(X_test)[:, 1]
        return {"model": self.name, "source": self.source,
                "accuracy": round(accuracy_score(y_test, y_pred), 3),
                "roc_auc": round(roc_auc_score(y_test, y_proba), 3)}

print("NoShowModelInterfaceV2 defined.")

NoShowModelInterfaceV2 defined.


In [43]:
baseline_wrapped_v2 = NoShowModelInterfaceV2(model, "LogisticRegression (Week 5 baseline)", "ML Engineering")
rf_wrapped_v2 = NoShowModelInterfaceV2(model_rf, "RandomForestClassifier (Week 6 comparison)", "ML Engineering")

sample = df.sample(3, random_state=RANDOM_SEED)
try:
    r1 = baseline_wrapped_v2.predict_risk(sample)
    r2 = rf_wrapped_v2.predict_risk(sample)
    backward_compat_ok = True
    backward_actual = f"Both models scored {len(r1)} and {len(r2)} rows respectively without error."
except Exception as e:
    backward_compat_ok = False
    backward_actual = f"CRASHED: {e}"

record_test(
    component="NoShowModelInterfaceV2 — backward compatibility",
    objective="Confirm the refactored interface still works for the original Week 6 models.",
    scenario="Wrap the Week 5/6 LogisticRegression and RandomForest models through NoShowModelInterfaceV2.",
    expected="Both score normally, identical behaviour to Week 6.",
    actual=backward_actual, passed=backward_compat_ok,
    evidence="This notebook, Week 7 refinement cell.",
)

print("--- Test F, AFTER fix ---")
try:
    baseline_wrapped_v2.predict_risk(bad_dtype_input)
    retest_f_ok = False
    retest_f_actual = "No error raised (unexpected)"
except ValueError as e:
    retest_f_ok = str(e).startswith(f"[{baseline_wrapped_v2.name}]") and "age" in str(e)
    retest_f_actual = f"{type(e).__name__}: {e}"
print(retest_f_actual)

record_test(
    component="NoShowModelInterfaceV2.predict_risk() — Week 7 fix",
    objective="Confirm the dtype fix produces a clear, named error instead of a raw internal exception.",
    scenario='Re-run Test F: score a batch where "age" contains text ("forty").',
    expected="A ValueError naming the interface and the 'age' column specifically.",
    actual=retest_f_actual, passed=retest_f_ok,
    issue="None — this row is the retest of the Test F issue above.",
    action="N/A", retest="PASSED — see Actual Result.",
    evidence="This notebook, Week 7 refinement cell.",
)

INFO:healthconnect.pipeline:[LogisticRegression (Week 5 baseline)] Scoring 3 appointment(s).
INFO:healthconnect.pipeline:[LogisticRegression (Week 5 baseline)] Returned 3 risk score(s).
INFO:healthconnect.pipeline:[RandomForestClassifier (Week 6 comparison)] Scoring 3 appointment(s).
INFO:healthconnect.pipeline:[RandomForestClassifier (Week 6 comparison)] Returned 3 risk score(s).


[PASS] NoShowModelInterfaceV2 — backward compatibility — Wrap the Week 5/6 LogisticRegression and RandomForest models through NoShowModelInterfaceV2.
--- Test F, AFTER fix ---
ValueError: [LogisticRegression (Week 5 baseline)] Column 'age' contains non-numeric value(s): ['forty']
[PASS] NoShowModelInterfaceV2.predict_risk() — Week 7 fix — Re-run Test F: score a batch where "age" contains text ("forty").


'PASS'

## 7.5 HC-POD Cross-Track Testing — Data Science (Alia Al-Qadri)

Alia's methodology matches this pipeline almost exactly (same missing-value handling,
same GroupShuffleSplit on patient_id, same 80/20 split with random_state=42, same
one-hot + scaling, same LogisticRegression baseline). The one real difference: she
engineered `booking_month` from `booking_date`. This pipeline doesn't use it. That's
the genuine dependency to test.

In [44]:
def engineer_features_v2(df):
    """engineer_features() + Alia's booking_month feature (Data Science spec)."""
    out = engineer_features(df)
    out["booking_month"] = pd.to_datetime(out["booking_date"]).dt.month.astype(str)
    return out

FEATURE_COLS_V2 = FEATURE_COLS + ["booking_month"]
CATEGORICAL_FEATURES_V2 = CATEGORICAL_FEATURES + ["booking_month"]

preprocessor_v2 = ColumnTransformer([
    ("numeric", numeric_transformer, NUMERIC_FEATURES),
    ("categorical", categorical_transformer, CATEGORICAL_FEATURES_V2),
])

train_df_v2 = engineer_features_v2(train_df)
test_df_v2 = engineer_features_v2(test_df)

X_train_v2, y_train_v2 = train_df_v2[FEATURE_COLS_V2], train_df_v2["no_show"]
X_test_v2, y_test_v2 = test_df_v2[FEATURE_COLS_V2], test_df_v2["no_show"]

model_v2 = Pipeline([
    ("preprocessor", preprocessor_v2),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")),
])
model_v2.fit(X_train_v2, y_train_v2)

y_pred_v2 = model_v2.predict(X_test_v2)
y_proba_v2 = model_v2.predict_proba(X_test_v2)[:, 1]

v2_metrics = {"accuracy": round(accuracy_score(y_test_v2, y_pred_v2), 3),
              "precision": round(precision_score(y_test_v2, y_pred_v2), 3),
              "recall": round(recall_score(y_test_v2, y_pred_v2), 3),
              "f1": round(f1_score(y_test_v2, y_pred_v2), 3),
              "roc_auc": round(roc_auc_score(y_test_v2, y_proba_v2), 3)}
baseline_full_metrics = {"accuracy": round(accuracy_score(y_test, y_pred), 3),
                         "precision": round(precision_score(y_test, y_pred), 3),
                         "recall": round(recall_score(y_test, y_pred), 3),
                         "f1": round(f1_score(y_test, y_pred), 3),
                         "roc_auc": round(roc_auc_score(y_test, y_proba), 3)}

print("Baseline (Week 6, no booking_month):", baseline_full_metrics)
print("With Alia's booking_month feature: ", v2_metrics)

record_test(
    component="Feature set — booking_month (Data Science spec)",
    objective="Determine whether Alia's independently-engineered booking_month feature changes performance.",
    scenario="Retrain LogisticRegression with booking_month added, identical split/hyperparameters otherwise.",
    expected="A measurable, reported change (either direction) in ROC-AUC vs the Week 6 baseline.",
    actual=v2_metrics, passed=True,
    issue="None — outcome reported either way, not assumed.",
    action="Adopt booking_month if ROC-AUC improves meaningfully; otherwise document as tested-and-rejected, "
           "same treatment as Week 6's tree-model comparison.",
    retest="N/A — first test of this feature.", track="Data Science",
    evidence="Slack thread with Alia Al-Qadri (#hc-pod-03), this notebook's cross-track testing cell.",
)

Baseline (Week 6, no booking_month): {'accuracy': 0.629, 'precision': 0.63, 'recall': 0.627, 'f1': 0.629, 'roc_auc': np.float64(0.683)}
With Alia's booking_month feature:  {'accuracy': 0.622, 'precision': 0.621, 'recall': 0.625, 'f1': 0.623, 'roc_auc': np.float64(0.682)}
[PASS] Feature set — booking_month (Data Science spec) — Retrain LogisticRegression with booking_month added, identical split/hyperparameters otherwise.


'PASS'

In [45]:
model_v2_wrapped_OLD = NoShowModelInterface(model_v2, "LogisticRegression + booking_month (Data Science spec)", "Data Science")

print("--- Wrapping the booking_month model in the WEEK 6 interface (expected to fail) ---")
try:
    model_v2_wrapped_OLD.predict_risk(df.sample(3, random_state=RANDOM_SEED))
    incompat_ok = False
    incompat_actual = "No error raised (unexpected — investigate)"
except Exception as e:
    incompat_ok = True
    incompat_actual = f"{type(e).__name__}: {e}"
print(incompat_actual)

record_test(
    component="NoShowModelInterface (Week 6 version) — cross-track compatibility",
    objective="Confirm whether the Week 6 interface can host a Data Science model with a different feature set.",
    scenario="Wrap model_v2 (LogisticRegression + booking_month) in the Week 6 interface and score a sample.",
    expected="FAILS — Week 6 interface hardcodes FEATURE_COLS/engineer_features, cannot select booking_month.",
    actual=incompat_actual, passed=incompat_ok,
    issue="Week 6 interface assumed every model shares the ML Engineering baseline's exact feature set. "
          "Alia's model needs one extra feature — a real integration gap.",
    action="Use NoShowModelInterfaceV2 (per-model feature_fn/feature_cols) instead.",
    retest="See next cell.", track="Data Science",
    evidence="This notebook, Week 7 cross-track compatibility cell.",
)

--- Wrapping the booking_month model in the WEEK 6 interface (expected to fail) ---
ValueError: columns are missing: {'booking_month'}
[PASS] NoShowModelInterface (Week 6 version) — cross-track compatibility — Wrap model_v2 (LogisticRegression + booking_month) in the Week 6 interface and score a sample.


'PASS'

In [46]:
model_v2_wrapped_NEW = NoShowModelInterfaceV2(
    model_v2, "LogisticRegression + booking_month (Data Science spec)", "Data Science",
    feature_fn=engineer_features_v2, feature_cols=FEATURE_COLS_V2,
)

print("--- Retest through NoShowModelInterfaceV2 ---")
try:
    v2_result = model_v2_wrapped_NEW.predict_risk(df.sample(3, random_state=RANDOM_SEED))
    retest_v2_ok = len(v2_result) == 3
    retest_v2_actual = v2_result.to_dict("records")
except Exception as e:
    retest_v2_ok = False
    retest_v2_actual = f"CRASHED: {e}"
print(retest_v2_actual)

record_test(
    component="NoShowModelInterfaceV2 — cross-track compatibility (retest)",
    objective="Confirm the refactored interface can host a Data Science model with a different feature set.",
    scenario="Re-run the previous test through NoShowModelInterfaceV2 with feature_fn=engineer_features_v2.",
    expected="Scores successfully — no crash, valid probabilities returned.",
    actual=retest_v2_actual, passed=retest_v2_ok,
    issue="None — retest of the compatibility gap identified above.", action="N/A",
    retest="PASSED — the model-integration point is now generalised, not fixed for one case only.",
    track="Data Science", evidence="This notebook, Week 7 cross-track compatibility cell.",
)

INFO:healthconnect.pipeline:[LogisticRegression + booking_month (Data Science spec)] Scoring 3 appointment(s).
INFO:healthconnect.pipeline:[LogisticRegression + booking_month (Data Science spec)] Returned 3 risk score(s).


--- Retest through NoShowModelInterfaceV2 ---
[{'appointment_id': 'HC-01502', 'no_show_probability': 0.6016, 'model_used': 'LogisticRegression + booking_month (Data Science spec)', 'risk_band': 'Medium'}, {'appointment_id': 'HC-02587', 'no_show_probability': 0.7609, 'model_used': 'LogisticRegression + booking_month (Data Science spec)', 'risk_band': 'High'}, {'appointment_id': 'HC-02654', 'no_show_probability': 0.3076, 'model_used': 'LogisticRegression + booking_month (Data Science spec)', 'risk_band': 'Low'}]
[PASS] NoShowModelInterfaceV2 — cross-track compatibility (retest) — Re-run the previous test through NoShowModelInterfaceV2 with feature_fn=engineer_features_v2.


'PASS'

7.6 Config/dependency review

config_review_lines = [
    "CONFIGURATION & DEPENDENCY REVIEW — Week 7",
    "",
    "No new dependencies required: NoShowModelInterfaceV2 uses only pandas and",
    "scikit-learn, already declared for Week 5/6. booking_month is derived with",
    "pandas' built-in datetime accessor -- no new library needed.",
    "",
    "config.yaml / requirements.txt: unchanged since Week 6, still accurate.",
]
print("\n".join(config_review_lines))

7.7 Testing & Validation Record

In [47]:
test_record_df = pd.DataFrame(test_log)
test_record_df.to_csv("HealthConnect_Week7_Testing_Record.csv", index=False)
from google.colab import files
files.download("HealthConnect_Week7_Testing_Record.csv")
test_record_df

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Component Tested,Testing Objective,Test/Scenario,Expected Result,Actual Result,Pass/Fail,Issue Identified,Action Taken,Retest Result,Collaborating Track,Evidence
0,Training step (LogisticRegression baseline),Confirm the documented setup (fixed RANDOM_SEE...,Retrain the Week 5/6 baseline from the same tr...,Identical metrics to the original run: {'accur...,"{'accuracy': 0.629, 'roc_auc': 0.683}",PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 reproducibility cell."
1,engineer_features() — prior_no_show_rate,Confirm the division-by-zero guard for first-t...,Run engineer_features() on a patient with prev...,"prior_no_show_rate == 0.0, not NaN/inf.","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 preprocessing test cell."
2,engineer_features() — reminder_channel,Confirm missing reminder_channel values are fi...,Run engineer_features() on rows where reminder...,"All such rows have reminder_channel == ""No Rem...",[No Reminder],PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 preprocessing test cell."
3,Preprocessing pipeline — OneHotEncoder(handle_...,Confirm an unseen categorical value doesn't cr...,"Score a batch where reminder_channel = ""Carrie...",Pipeline handles it silently and returns valid...,"No crash — probabilities returned: [0.64, 0.78...",PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 preprocessing test cell."
4,NoShowModelInterface.predict_risk(),Confirm the interface works on the smallest va...,Score a single-row input.,Returns exactly 1 row with a valid probability...,"[{'appointment_id': 'HC-01502', 'no_show_proba...",PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 input-validation test cell."
5,NoShowModelInterface.predict_risk(),Confirm unrecognised extra columns don't break...,Score a batch with two extra columns not used ...,Extra columns are ignored; scoring succeeds no...,"3 rows returned, extra columns ignored.",PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 input-validation test cell."
6,NoShowModelInterface.predict_risk() — Week 6 v...,Confirm malformed numeric input fails with a c...,"Score a batch where ""age"" contains text (""fort...",A clear ValueError naming the interface and th...,ValueError: Cannot use median strategy with no...,FAIL,Week 6 interface has no dtype validation — a b...,Refactor the interface (below) to validate num...,"See retest after the refinement, next section.",ML Engineering,"This notebook, Week 7 input-validation test cell."
7,NoShowModelInterfaceV2 — backward compatibility,Confirm the refactored interface still works f...,Wrap the Week 5/6 LogisticRegression and Rando...,"Both score normally, identical behaviour to We...",Both models scored 3 and 3 rows respectively w...,PASS,None — validation successful,No action required,N/A,ML Engineering,"This notebook, Week 7 refinement cell."
8,NoShowModelInterfaceV2.predict_risk() — Week 7...,"Confirm the dtype fix produces a clear, named ...","Re-run Test F: score a batch where ""age"" conta...",A ValueError naming the interface and the 'age...,ValueError: [LogisticRegression (Week 5 baseli...,PASS,None — this row is the retest of the Test F is...,N/A,PASSED — see Actual Result.,ML Engineering,"This notebook, Week 7 refinement cell."
9,Feature set — booking_month (Data Science spec),Determine whether Alia's independently-enginee...,Retrain LogisticRegression with booking_month ...,"A measurable, reported change (either directio...","{'accuracy': 0.622, 'precision': 0.621, 'recal...",PASS,"None — outcome reported either way, not assumed.",Adopt booking_month if ROC-AUC improves meanin...,N/A — first test of this feature.,Data Science,"Slack thread with Alia Al-Qadri (#hc-pod-03), ..."


7.8 Remaining issues & Week 8 readiness

week8_readiness_lines = [
    "REMAINING ISSUES & WEEK 8 READINESS -- ML Engineering Track",
    "",
    "Resolved this week:",
    "- No dtype validation on numeric columns -> resolved (clear, named error now).",
    "- Interface couldn't host a model with a different feature set -> resolved",
    "  (feature_fn/feature_cols are now per-model, not hardcoded globals).",
    "",
    "Still open going into Week 8:",
    "- Alia's actual fitted model/artifact has still not been received -- she has",
    "  now confirmed her full methodology but offered a processed dataset + notebook",
    "  rather than a serialized model. The booking_month reproduction above is the",
    "  best available stand-in until a real artifact arrives.",
    "- No fairness evaluation has been run on the model's use of gender/age features.",
    "- The Sunday-appointment / Knowledge Base inconsistency from Week 5 remains unresolved.",
    "- [Fill in once you've seen the real result above:] whether booking_month should be",
    "  adopted permanently, or documented as tested-and-not-adopted like Week 6's tree models.",
]
print("\n".join(week8_readiness_lines))